# AorticTurb — Reproducible Google Colab Study Runner

This notebook is a thin orchestration layer over **AorticTurb**. It does not reimplement the scientific calculations.

Locked AorticTurb commit: `cfc9a3c80d946e8616bb3c57e56be25bd40fc1ca`

Results are written persistently under `<Google Drive project root>/results/`. Raw Drive data are never modified.

In [ ]:
import sys, subprocess
from pathlib import Path
from importlib import metadata as importlib_metadata

AORTICTURB_REPO = 'https://github.com/khalid-saqr/AorticTurb.git'
AORTICTURB_COMMIT = 'cfc9a3c80d946e8616bb3c57e56be25bd40fc1ca'
EXPECTED_AORTICTURB_VERSION = '0.2.0'

if sys.version_info[:2] not in {(3,10), (3,11), (3,12)}:
    raise RuntimeError(f'UNSUPPORTED_PYTHON: {sys.version.split()[0]}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', f'git+{AORTICTURB_REPO}@{AORTICTURB_COMMIT}'], check=True)

import aorticturb
from aorticturb.workflow import run_all
from aorticturb.data.verify import verify_dataset
from aorticturb.validation.startup_gate import run_startup_gate
from aorticturb.config import load_config

if aorticturb.__version__ != EXPECTED_AORTICTURB_VERSION:
    raise RuntimeError(f'INSTALL_FAILED: expected {EXPECTED_AORTICTURB_VERSION}, got {aorticturb.__version__}')

print('AorticTurb', aorticturb.__version__)
print('Commit', AORTICTURB_COMMIT)

In [ ]:
import os, shutil, platform, hashlib, json, uuid
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

# EDIT ONLY THIS PATH IF YOUR PROJECT IS STORED ELSEWHERE.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study').resolve()
DRIVE_RESULTS_ROOT = DRIVE_PROJECT_ROOT / 'results'
LOCAL_ROOT = Path('/content/aorticturb_input')
PHASES = ('t018','t021','t024')
REALIZATIONS = tuple(f'B{i:05d}.npy' for i in range(1,17))

def fail(code, message):
    raise RuntimeError(f'{code}: {message}')

def sha256_file(path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def parse_ledger(path):
    records = {}
    for raw in Path(path).read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            fail('SOURCE_PROVENANCE_FAILED', f'malformed checksum line: {line}')
        records[parts[1].strip().lstrip('*')] = parts[0].lower()
    return records

def copy_verified(src, dst, expected=None):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    h = hashlib.sha256()
    with src.open('rb') as fin, dst.open('wb') as fout:
        while True:
            block = fin.read(16*1024*1024)
            if not block:
                break
            h.update(block)
            fout.write(block)
    streamed = h.hexdigest()
    if expected and streamed.lower() != expected.lower():
        dst.unlink(missing_ok=True)
        fail('COPY_HASH_MISMATCH', f'{src}: expected {expected}, got {streamed}')
    if sha256_file(dst) != streamed:
        dst.unlink(missing_ok=True)
        fail('COPY_HASH_MISMATCH', f'local write verification failed: {dst}')
    return streamed

if not DRIVE_PROJECT_ROOT.is_dir():
    fail('DATASET_ROOT_INVALID', 'edit DRIVE_PROJECT_ROOT to the mounted project directory')
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
probe = DRIVE_RESULTS_ROOT / '.write_test'
probe.write_text('ok\n', encoding='utf-8')
probe.unlink()

free_gib = shutil.disk_usage('/content').free / (1024**3)
if free_gib < 8:
    fail('INSUFFICIENT_LOCAL_STORAGE', f'only {free_gib:.1f} GiB free')

velocity_root = DRIVE_PROJECT_ROOT / '01_raw_data' / 'M_medium_root' / 'AAo'
for phase in PHASES:
    observed = sorted(p.name for p in (velocity_root/phase).glob('*.npy'))
    if observed != list(REALIZATIONS):
        fail('DATASET_ROOT_INVALID', f'{phase} does not contain exactly B00001–B00016')

geometry_sources = {
    'M': DRIVE_PROJECT_ROOT/'02_metadata_geometry/M_medium_root/original_geometry/M_aortic_root.STL',
    'S': DRIVE_PROJECT_ROOT/'02_metadata_geometry/S_small_root/original_geometry/S_aortic_root.STL',
    'L': DRIVE_PROJECT_ROOT/'02_metadata_geometry/L_large_root/original_geometry/L_aortic_root.STL',
    'NS': DRIVE_PROJECT_ROOT/'02_metadata_geometry/NS_non_sinus_root/original_geometry/NS_aortic_root.STL',
}
for p in geometry_sources.values():
    if not p.is_file():
        fail('DATASET_ROOT_INVALID', f'missing geometry {p}')

historical_ledger = DRIVE_PROJECT_ROOT/'00_provenance/checksums/final_acquired_SHA256SUMS.txt'
if not historical_ledger.is_file():
    fail('SOURCE_PROVENANCE_FAILED', 'missing final_acquired_SHA256SUMS.txt')
historical = parse_ledger(historical_ledger)
by_basename = defaultdict(set)
for old_path, digest in historical.items():
    by_basename[Path(old_path).name].add(digest)

if LOCAL_ROOT.exists():
    if not str(LOCAL_ROOT).startswith('/content/'):
        fail('LOCAL_STAGING_INCOMPLETE', 'refusing to delete non-/content path')
    shutil.rmtree(LOCAL_ROOT)
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)

for name in ('METADATA.json','AGENTS.md'):
    src = DRIVE_PROJECT_ROOT/name
    if not src.is_file():
        fail('DATASET_ROOT_INVALID', f'missing {src}')
    copy_verified(src, LOCAL_ROOT/name)

prov = DRIVE_PROJECT_ROOT/'00_provenance'
for src in sorted(p for p in prov.rglob('*') if p.is_file()):
    copy_verified(src, LOCAL_ROOT/src.relative_to(DRIVE_PROJECT_ROOT))

for phase in PHASES:
    for name in REALIZATIONS:
        rel = Path('01_raw_data')/'M_medium_root'/'AAo'/phase/name
        expected = historical.get(rel.as_posix())
        if expected is None:
            fail('SOURCE_PROVENANCE_FAILED', f'ledger omits {rel.as_posix()}')
        copy_verified(DRIVE_PROJECT_ROOT/rel, LOCAL_ROOT/rel, expected)

geometry_dest = {
    'M': Path('02_metadata_geometry/M_medium_root/original_geometry/M_aortic_root.STL'),
    'S': Path('02_metadata_geometry/S_small_root/original_geometry/S_aortic_root.STL'),
    'L': Path('02_metadata_geometry/L_large_root/original_geometry/L_aortic_root.STL'),
    'NS': Path('02_metadata_geometry/NS_non_sinus_root/original_geometry/NS_aortic_root.STL'),
}
for key, src in geometry_sources.items():
    candidates = by_basename[src.name]
    if len(candidates) != 1:
        fail('SOURCE_PROVENANCE_FAILED', f'cannot uniquely resolve {src.name}')
    copy_verified(src, LOCAL_ROOT/geometry_dest[key], next(iter(candidates)))

rows = []
for p in sorted(x for x in LOCAL_ROOT.rglob('*') if x.is_file()):
    if p.name == 'SHA256SUMS.txt' and p.parent == LOCAL_ROOT:
        continue
    rows.append(f'{sha256_file(p)}  {p.relative_to(LOCAL_ROOT).as_posix()}')
package_ledger = LOCAL_ROOT/'SHA256SUMS.txt'
package_ledger.write_text('\n'.join(rows)+'\n', encoding='utf-8')
package_ledger_sha256 = sha256_file(package_ledger)

verification = verify_dataset(LOCAL_ROOT, verify_checksums=True, deep_content_check=True)
if not verification.ok:
    fail('AORTICTURB_VERIFY_FAILED', json.dumps(verification.to_dict()))
startup = run_startup_gate(LOCAL_ROOT, verify_checksums=True, deep_content_check=True)
if not startup.passed:
    fail('STARTUP_GATE_FAILED', json.dumps(startup.to_dict()))

config = load_config(None)
config_sha256 = hashlib.sha256(json.dumps(config, sort_keys=True, separators=(',',':')).encode()).hexdigest()
execution_id = 'colab_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_') + uuid.uuid4().hex[:8]
record_dir = DRIVE_RESULTS_ROOT/'colab_runs'/execution_id
record_dir.mkdir(parents=True, exist_ok=False)
manifest = {
    'execution_id': execution_id,
    'status': 'RUNNING',
    'started_utc': datetime.now(timezone.utc).isoformat(),
    'aorticturb_commit': AORTICTURB_COMMIT,
    'aorticturb_version': aorticturb.__version__,
    'python': sys.version,
    'platform': platform.platform(),
    'drive_project_root': str(DRIVE_PROJECT_ROOT),
    'drive_results_root': str(DRIVE_RESULTS_ROOT),
    'local_package_ledger_sha256': package_ledger_sha256,
    'config_sha256': config_sha256,
    'software_versions': {name: importlib_metadata.version(name) for name in ['aorticturb','numpy','scipy','pandas','matplotlib','PyYAML','requests','typer','rich']},
}
(record_dir/'execution_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True)+'\n', encoding='utf-8')
(record_dir/'config_snapshot.json').write_text(json.dumps(config, indent=2, sort_keys=True)+'\n', encoding='utf-8')
print('Input staging and all AorticTurb gates: PASS')
print('Results root:', DRIVE_RESULTS_ROOT)

In [ ]:
try:
    study_summary = run_all(LOCAL_ROOT, DRIVE_RESULTS_ROOT, config_path=None)
except Exception as exc:
    manifest['status'] = 'STUDY_EXECUTION_FAILED'
    manifest['failed_utc'] = datetime.now(timezone.utc).isoformat()
    manifest['error_type'] = type(exc).__name__
    manifest['error'] = str(exc)
    (record_dir/'execution_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True)+'\n', encoding='utf-8')
    raise

print('Study ID:', study_summary['study_id'])

In [ ]:
import pandas as pd

study_id = study_summary.get('study_id')
summary_path = DRIVE_RESULTS_ROOT/'05_results'/'study_runs'/study_id/'study_summary.json'
if not summary_path.is_file():
    fail('OUTPUT_VALIDATION_FAILED', f'missing {summary_path}')

expected_stages = {'decomposition','regime','kolmogorov_obukhov','taylor','anisotropy','diagnostics','classification'}
stages = study_summary.get('stages', {})
if set(stages) != expected_stages:
    fail('OUTPUT_VALIDATION_FAILED', f'unexpected stages: {sorted(stages)}')

for stage, path_text in stages.items():
    run_dir = Path(path_text)
    rm = run_dir/'run_manifest.json'
    ih = run_dir/'input_hashes.txt'
    if not run_dir.is_dir() or not rm.is_file() or not ih.is_file():
        fail('OUTPUT_VALIDATION_FAILED', f'incomplete {stage} run bundle')
    run_manifest = json.loads(rm.read_text(encoding='utf-8'))
    if run_manifest.get('status') != 'COMPLETE' or not run_manifest.get('outputs'):
        fail('OUTPUT_VALIDATION_FAILED', f'{stage} did not finalize correctly')

classification_dir = Path(stages['classification'])
classification = json.loads((classification_dir/'classification_summary.json').read_text(encoding='utf-8'))
phase_classifications = classification.get('phase_classifications', {})
if set(phase_classifications) != {'t018','t021','t024'}:
    fail('OUTPUT_VALIDATION_FAILED', 'missing phase classifications')

manifest['status'] = 'COMPLETE'
manifest['completed_utc'] = datetime.now(timezone.utc).isoformat()
manifest['study_id'] = study_id
manifest['study_summary_path'] = str(summary_path)
manifest['phase_classifications'] = phase_classifications
(record_dir/'execution_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True)+'\n', encoding='utf-8')
completion = {
    'status': 'COMPLETE',
    'execution_id': execution_id,
    'study_id': study_id,
    'validated_utc': datetime.now(timezone.utc).isoformat(),
    'aorticturb_commit': AORTICTURB_COMMIT,
    'config_sha256': config_sha256,
    'study_summary_path': str(summary_path),
}
(record_dir/'COLAB_RUN_COMPLETE.json').write_text(json.dumps(completion, indent=2, sort_keys=True)+'\n', encoding='utf-8')

rows = []
for phase in ('t018','t021','t024'):
    item = phase_classifications[phase]
    rows.append({'phase': phase, 'classification': item.get('label'), 'rationale': ' | '.join(item.get('rationale', []))})
display(pd.DataFrame(rows))
print('Study summary:', summary_path)
print('Completion sentinel:', record_dir/'COLAB_RUN_COMPLETE.json')

if LOCAL_ROOT.exists():
    if not str(LOCAL_ROOT).startswith('/content/'):
        fail('LOCAL_CLEANUP_BLOCKED', 'refusing to remove path outside /content')
    shutil.rmtree(LOCAL_ROOT)
print('Local scratch cleaned; persistent results remain in', DRIVE_RESULTS_ROOT)